In [1]:
#Importing Libraries

from collections import Counter
import json
from pathlib import Path

In [2]:
# Confirming and setting project path

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "Data" / "Raw" #Contains datasets csv
print(BASE_DIR)
print(RAW_DIR)

D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Raw


In [3]:
# Load json file

json_path = RAW_DIR / "synthetic_notes.json"

with open(json_path, "r", encoding="utf-8") as file:
    data = json.load(file)

In [4]:
#Convert to bio records, add tag = 'O'

from nltk.tokenize import word_tokenize

def convert_record_to_bio(record):
    text = record["text"]
    tokens = word_tokenize(text)

    bio_rows = []
    current_pos = 0

    for token in tokens:
        # Find token position starting from current_pos
        start = text.find(token, current_pos)
        end = start + len(token)

        tag = "O"

        # Check if token falls inside any entity span
        for entity in record["entities"]:
            if start >= entity["start"] and end <= entity["end"]:
                if start == entity["start"]:
                    tag = f"B-{entity['label']}"
                else:
                    tag = f"I-{entity['label']}"
                break

        bio_rows.append((token, tag))
        current_pos = end

    return bio_rows


# Verify

bio_rows = convert_record_to_bio(data[0])

for row in bio_rows:
    print(row)

('Known', 'O')
('case', 'O')
('of', 'O')
('Influenza', 'B-DISEASE')
('.', 'O')
('Advised', 'O')
('to', 'O')
('take', 'O')
('Budesonide', 'B-MEDICATION')
('250', 'B-DOSAGE')
('mg', 'I-DOSAGE')
('once', 'I-DOSAGE')
('daily', 'I-DOSAGE')
('.', 'O')


In [5]:
# Process All Records

all_rows = []

for record in data:

    bio_rows = convert_record_to_bio(record)

    for token, tag in bio_rows:

        all_rows.append({
            "record_id": record["id"],
            "token": token,
            "tag": tag
        })

In [6]:
# Convert to DataFrame

import pandas as pd

df = pd.DataFrame(all_rows)

In [7]:
project_root = r'D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data'
print(project_root)

# Save Dataset

project_root = Path(project_root)   # convert string → Path

PROCESSED_DIR = project_root / "Processed"
output_file = PROCESSED_DIR / "bio_dataset.csv"

df.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(df))

D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data
Saved: D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Processed\bio_dataset.csv
Rows: 77527


## Verify

In [8]:
print(df.head(5))

print("\nTag Counts:")
print(df["tag"].value_counts())
print('\n')
print("Total Rows:", len(df))
print("Unique Records:", df["record_id"].nunique())
print('\n')
df[:5]

   record_id      token        tag
0          1      Known          O
1          1       case          O
2          1         of          O
3          1  Influenza  B-DISEASE
4          1          .          O

Tag Counts:
tag
O               45171
I-DOSAGE        10799
B-DISEASE        4558
B-MEDICATION     4175
B-DOSAGE         4175
B-SYMPTOM        3186
I-SYMPTOM        2101
I-DISEASE        1967
B-PROCEDURE       825
I-PROCEDURE       570
Name: count, dtype: int64


Total Rows: 77527
Unique Records: 5000




,record_id,token,tag
0,1,Known,O
1,1,case,O
2,1,of,O
3,1,Influenza,B-DISEASE
4,1,.,O
